# VisRAG chunks manual retrieval demo

Мини-блокнот проверяет новый контур `visrag` на реальном коде проекта: профиль данных Iris → анализ запроса → загрузка chunk-корпуса и embeddings → ручной retrieval → полный `VisRAGService.invoke`.

Перед запуском должны существовать файлы:

    rag_corpus/runtime/guidance_chunks.jsonl
    rag_corpus/runtime/guidance_chunk_embeddings.jsonl

Если их нет, сначала выполнить:

    python scripts/rag_corpus/export_guidance_chunks.py
    python scripts/rag_corpus/build_visrag_embeddings.py --provider ollama --model <embedding-model> --base-url http://localhost:11434

In [1]:
import sys
from pathlib import Path

from rich import print as rprint

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = next(parent for parent in Path.cwd().resolve().parents if (parent / "src").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.application.project_config import load_project_config
from src.infrastructure.runtime import RuntimeContext
from src.llm.factory import build_chat_model
from src.services.data_profiler import DataProfilerService
from src.services.query_request_analyzer import QueryRequestAnalyzerService
from src.services.visrag import VisRAGService
from src.visrag_core import create_visrag_store
from src.visrag_core.embeddings import build_embedding_model, cosine
from src.visrag_core.query_builder import build_visrag_query

CONFIG_PATH = PROJECT_ROOT / "ui" / "config" / "app" / "project-gemma4.toml"
DATA_PATH = PROJECT_ROOT / "demo_data" / "Iris.csv"
QUERY = "Show the relationship between sepal length and petal length by iris species."
TOP_K = 8

project_config = load_project_config(CONFIG_PATH)
runtime = RuntimeContext(
    settings=project_config.settings,
    reasoning_llm=build_chat_model(project_config.reasoning_model),
)
runtime.settings.visrag_corpus_root = PROJECT_ROOT / "rag_corpus" / "runtime"

rprint({
    "project_root": str(PROJECT_ROOT),
    "config": str(CONFIG_PATH),
    "data": str(DATA_PATH),
    "query": QUERY,
    "visrag_options": project_config.settings.visrag_runtime_options(),
})

{
    'project_root': 'D:\\programming\\projects\\ViRAGE',
    'config': 'D:\\programming\\projects\\ViRAGE\\ui\\config\\app\\project-gemma4.toml',
    'data': 'D:\\programming\\projects\\ViRAGE\\demo_data\\Iris.csv',
    'query': 'Show the relationship between sepal length and petal length by iris species.',
    'visrag_options': {
        'enabled': True,
        'corpus_root': WindowsPath('D:/programming/projects/ViRAGE/rag_corpus/runtime'),
        'store_backend': 'jsonl',
        'top_k_chunks': 8,
        'embedding_provider': 'ollama',
        'embedding_model': 'nomic-embed-text',
        'embedding_base_url': 'http://localhost:11434'
    }
}

In [2]:
data_profile = DataProfilerService().invoke(str(DATA_PATH), runtime=runtime)

rprint({
    "row_count": data_profile.row_count,
    "col_count": data_profile.col_count,
    "columns": [
        {
            "name": column.name,
            "dtype": column.dtype,
            "role": column.role,
            "unique_count": column.unique_count,
            "min": column.min_value,
            "max": column.max_value,
        }
        for column in data_profile.columns
    ],
    "quality_notes": data_profile.quality_notes,
})

{
    'row_count': 150,
    'col_count': 6,
    'columns': [
        {'name': 'Id', 'dtype': 'numeric', 'role': 'identifier', 'unique_count': 150, 'min': 1.0, 'max': 150.0},
        {
            'name': 'SepalLengthCm',
            'dtype': 'numeric',
            'role': 'measure',
            'unique_count': 35,
            'min': 4.3,
            'max': 7.9
        },
        {
            'name': 'SepalWidthCm',
            'dtype': 'numeric',
            'role': 'measure',
            'unique_count': 23,
            'min': 2.0,
            'max': 4.4
        },
        {
            'name': 'PetalLengthCm',
            'dtype': 'numeric',
            'role': 'measure',
            'unique_count': 43,
            'min': 1.0,
            'max': 6.9
        },
        {
            'name': 'PetalWidthCm',
            'dtype': 'numeric',
            'role': 'measure',
            'unique_count': 22,
            'min': 0.1,
            'max': 2.5
        },
        {
            'name': 'Species',
            'dtype': 'categorical',
            'role': 'dimension',
            'unique_count': 3,
            'min': None,
            'max': None
        }
    ],
    'quality_notes': [
        "Column 'Id' looks like an identifier.",
        "Column 'SepalWidthCm' has 4 potential numeric outliers (2.7%)."
    ]
}

In [3]:
query_analysis = QueryRequestAnalyzerService().invoke(
    query=QUERY,
    user_context={"notebook": "visrag_chunks_manual_retrieval"},
    data_profile=data_profile,
    runtime=runtime,
)

rprint({
    "normalized_query": query_analysis.normalized_query,
    "analysis_task": query_analysis.analysis_task,
    "recommended_chart_family": query_analysis.recommended_chart_family,
    "selected_fields": query_analysis.selected_fields,
    "field_bindings": {key: value.model_dump() for key, value in query_analysis.field_bindings.items()},
    "query_variants": [variant.model_dump() for variant in query_analysis.query_variants],
})

{
    'normalized_query': 'Show the relationship between sepal length and petal length by iris species.',
    'analysis_task': 'correlation',
    'recommended_chart_family': 'auto',
    'selected_fields': ['SepalLengthCm', 'PetalLengthCm', 'Species'],
    'field_bindings': {
        'x': {
            'field': 'SepalLengthCm',
            'role': 'measure_axis',
            'confidence': 1.0,
            'rationale': 'Sepal length is requested as one of the primary variables for the relationship.'
        },
        'y': {
            'field': 'PetalLengthCm',
            'role': 'measure_axis',
            'confidence': 1.0,
            'rationale': 'Petal length is requested as the other primary variable for the relationship.'
        },
        'color': {
            'field': 'Species',
            'role': 'dimension_axis',
            'confidence': 1.0,
            'rationale': "The request specifies 'by iris species', implying a grouping or color encoding."
        }
    },
    'query_variants': [
        {
            'kind': 'canonical',
            'text': 'Show the relationship between sepal length and petal length by iris species.',
            'confidence': 1.0,
            'source': 'llm'
        },
        {
            'kind': 'chart_pattern_retrieval',
            'text': 'scatter plot sepal length vs petal length colored by species',
            'confidence': 0.9,
            'source': 'llm'
        }
    ]
}

In [4]:
opts = runtime.settings.visrag_runtime_options()
store = create_visrag_store(
    backend=str(opts["store_backend"]),
    uri=opts["corpus_root"],
)
chunks = store.load_chunks()
embeddings = store.load_embeddings()
missing_embeddings = [chunk.chunk_id for chunk in chunks if chunk.chunk_id not in embeddings]

if missing_embeddings:
    raise RuntimeError(
        "Embeddings are incomplete. Run: "
        "python scripts\\rag_corpus\\build_visrag_embeddings.py "
        f"Missing={len(missing_embeddings)} first={missing_embeddings[0]}"
    )

rprint({
    "backend": store.backend_name,
    "corpus_uri": store.corpus_uri,
    "signature": store.corpus_signature(),
    "chunks": len(chunks),
    "embeddings": len(embeddings),
    "source_counts": {
        source_id: sum(1 for chunk in chunks if chunk.source_id == source_id)
        for source_id in sorted({chunk.source_id for chunk in chunks})
    },
})

{
    'backend': 'jsonl',
    'corpus_uri': 'D:\\programming\\projects\\ViRAGE\\rag_corpus\\runtime',
    'signature': {
        'backend': 'jsonl',
        'uri': 'D:\\programming\\projects\\ViRAGE\\rag_corpus\\runtime',
        'chunks_path': 'D:/programming/projects/ViRAGE/rag_corpus/runtime/guidance_chunks.jsonl',
        'embeddings_path': 'D:/programming/projects/ViRAGE/rag_corpus/runtime/guidance_chunk_embeddings.jsonl',
        'chunks_exists': True,
        'embeddings_exists': True,
        'hash': '23e3e395456b40b2e92d6d675931722bb080c2326a95750e06c21a6bbe5964d7',
        'cache_key': 
'jsonl:D:\\programming\\projects\\ViRAGE\\rag_corpus\\runtime\\guidance_chunks.jsonl:D:\\programming\\projects\\ViR
AGE\\rag_corpus\\runtime\\guidance_chunk_embeddings.jsonl:23e3e395456b40b2e92d6d675931722bb080c2326a95750e06c21a6bb
e5964d7'
    },
    'chunks': 661,
    'embeddings': 661,
    'source_counts': {
        'from_data_to_viz': 169,
        'uk_analysis_colours': 36,
        'uk_charts_checklist': 21,
        'urban_institute_style_guide': 68,
        'wilke_fundamentals': 367
    }
}

In [5]:
retrieval_query = build_visrag_query(query_analysis, data_profile)

rprint({
    "retrieval_query": retrieval_query,
})

{
    'retrieval_query': 'Show the relationship between sepal length and petal length by iris species. correlation 
auto SepalLengthCm PetalLengthCm Species x SepalLengthCm measure_axis y PetalLengthCm measure_axis color Species 
dimension_axis Show the relationship between sepal length and petal length by iris species. scatter plot sepal 
length vs petal length colored by species Id SepalLengthCm SepalWidthCm PetalLengthCm PetalWidthCm Species'
}

In [6]:
embedder = build_embedding_model(
    provider=str(opts["embedding_provider"]),
    model=str(opts["embedding_model"]),
    base_url=str(opts["embedding_base_url"] or "") or None,
)
query_vector = list(embedder.embed_query(retrieval_query))

ranked = []
for chunk in chunks:
    score = cosine(query_vector, embeddings[chunk.chunk_id])
    if score > 0:
        ranked.append((score, chunk))
ranked = sorted(ranked, key=lambda item: (-item[0], item[1].source_id, item[1].chunk_id))[:TOP_K]

rprint([
    {
        "rank": index,
        "score": round(score, 6),
        "source_id": chunk.source_id,
        "source_kind": chunk.source_kind,
        "title": chunk.title,
        "source_path": chunk.source_path,
        "text_preview": chunk.text[:450],
    }
    for index, (score, chunk) in enumerate(ranked, start=1)
])

[
    {
        'rank': 1,
        'score': 0.762883,
        'source_id': 'wilke_fundamentals',
        'source_kind': 'web_guidance',
        'title': 'avoid line drawings',
        'source_path': 'wilke_fundamentals/dataviz_avoid-line-drawings.txt',
        'text_preview': 'lines for all three distributions.\nFigure 25.3: Density estimates of the sepal lengths of
three different iris species.\nThe broken line styles used for versicolor and virginica detract from the perception
that the areas under the curves are distinct from the areas above them.\nWe can attempt to address the problem of 
porous boundaries by using colored lines rather than dashed lines (Figure 25.4 ).\nHowever, the density areas in 
the resulting plot s'
    },
    {
        'rank': 2,
        'score': 0.727156,
        'source_id': 'wilke_fundamentals',
        'source_kind': 'web_guidance',
        'title': 'visualizing associations',
        'source_path': 'wilke_fundamentals/dataviz_visualizing-associations.txt',
        'text_preview': 'se equal.\nFigure 12.4: All-against-all scatter plot matrix of head length, body mass, and
skull size, for 123 blue jays.\nThis figure shows the exact same data as Figure 12.2 .\nHowever, because we are 
better at judging position than symbol size, correlations between skull size and the other two variables are easier
to perceive in the pairwise scatter plots than in Figure 12.2 .\nData source: Keith Tarvin, Oberlin College\n## 
12.2 Correlograms\nWhen w'
    },
    {
        'rank': 3,
        'score': 0.723779,
        'source_id': 'wilke_fundamentals',
        'source_kind': 'web_guidance',
        'title': 'visualizing associations',
        'source_path': 'wilke_fundamentals/dataviz_visualizing-associations.txt',
        'text_preview': '# Fundamentals of Data Visualization\nSource URL: 
https://clauswilke.com/dataviz/visualizing-associations.html\n# 12 Visualizing associations among two or more 
quantitative variables\nMany datasets contain two or more quantitative variables, and we may be interested in how 
these variables relate to each other.\nFor example, we may have a dataset of quantiative measurements of different 
animals, such as the animals’ height, weight, length, and daily e'
    },
    {
        'rank': 4,
        'score': 0.714485,
        'source_id': 'wilke_fundamentals',
        'source_kind': 'web_guidance',
        'title': 'redundant coding',
        'source_path': 'wilke_fundamentals/dataviz_redundant-coding.txt',
        'text_preview': 'al groups of data are frequently designed such that the points representing different 
groups differ only in their color.\nAs an example, consider Figure 20.1 , which shows the sepal width versus the 
sepal length of three different Iris species. (Sepals are the outer leafs of flowers in flowering plants.) The 
points representing the different species differ in their colors, but otherwise all points look exactly the 
same.\nEven though this figure con'
    },
    {
        'rank': 5,
        'score': 0.714219,
        'source_id': 'from_data_to_viz',
        'source_kind': 'web_guidance',
        'title': 'graph parallel',
        'source_path': 'from_data_to_viz/graph_parallel.txt',
        'text_preview': '# Parallel coordinates plot – from Data to Viz\nSource URL: 
https://www.data-to-viz.com/graph/parallel.html\n# Definition\nParallel plot or parallel coordinates plot allows to
compare the feature of several individual observations ( series ) on a set of numeric variables.\nEach vertical bar
represents a variable and often has its own scale. (The units can even be different).\nValues are then plotted as 
series of lines connected across each axis.\nThe ì'
    },
    {
        'rank': 6,
        'score': 0.709608,
        'source_id': 'wilke_fundamentals',
        'source_kind': 'web_guidance',
        'title': 'visualizing associations',
        'source_path': 'wilke_fundamentals/dataviz_visualizing-associations.txt',
        'text_preview': 'ween data

In [7]:
visrag_result = VisRAGService().invoke(
    query_analysis=query_analysis,
    data_profile=data_profile,
    runtime=runtime,
)

rprint({
    "retrieval_strategy": visrag_result.retrieval_strategy,
    "diagnostics": visrag_result.diagnostics.model_dump(),
    "generation_guidance": visrag_result.generation_guidance.model_dump(),
})

{
    'retrieval_strategy': 'chunk_guidance:jsonl:precomputed_embeddings',
    'diagnostics': {
        'warnings': [],
        'retrieved_count': 8,
        'retrieved_count_by_type': {'web_guidance': 8},
        'corpus_backend': 'jsonl',
        'corpus_uri': 'D:\\programming\\projects\\ViRAGE\\rag_corpus\\runtime',
        'corpus_hash': '23e3e395456b40b2e92d6d675931722bb080c2326a95750e06c21a6bbe5964d7'
    },
    'generation_guidance': {
        'applicable_rules': [
            'Use a scatter plot to visualize the relationship between two quantitative variables (SepalLengthCm and
PetalLengthCm).',
            "Map the categorical dimension 'Species' to color to distinguish between the three iris species.",
            'Apply a small amount of jitter to point positions to prevent overplotting where data points 
intermingle.'
        ],
        'avoid': [
            'Avoid using only color to distinguish groups if the colors are not distinct (e.g., green and blue), as
this can make the chart difficult to read.',
            'Avoid using bubble size to encode quantitative variables when position is available, as size 
differences are harder to perceive than position differences.',
            'Avoid using open circles, triangles, or crosses (line drawings) for points in scatter plots.'
        ],
        'quality_checks': [
            "Ensure that the 'Species' encoding provides a clear visual separation between Iris-setosa, 
Iris-versicolor, and Iris-virginica.",
            'Verify that the x-axis is mapped to SepalLengthCm and the y-axis to PetalLengthCm (or vice versa) to 
correctly show the correlation.'
        ],
        'feedback_warnings': [
            'Warning: If species distributions overlap significantly, relying solely on color may be insufficient; 
consider redundant coding or jittering to improve legibility.'
        ],
        'source_refs': [
            {
                'chunk_id': 'chunk_3686b1e940745d099ab6cb818c340ecd',
                'source_id': 'wilke_fundamentals',
                'source_name': 'wilke fundamentals',
                'title': 'avoid line drawings',
                'score': 0.762883,
                'source_path': 'wilke_fundamentals/dataviz_avoid-line-drawings.txt',
                'url': None
            },
            {
                'chunk_id': 'chunk_4fd05b1ef0875c81a8cb81659bf740d6',
                'source_id': 'wilke_fundamentals',
                'source_name': 'wilke fundamentals',
                'title': 'visualizing associations',
                'score': 0.727156,
                'source_path': 'wilke_fundamentals/dataviz_visualizing-associations.txt',
                'url': None
            },
            {
                'chunk_id': 'chunk_7d50fa5b16da5475bb26d4385a699bf8',
                'source_id': 'wilke_fundamentals',
                'source_name': 'wilke fundamentals',
                'title': 'visualizing associations',
                'score': 0.723779,
                'source_path': 'wilke_fundamentals/dataviz_visualizing-associations.txt',
                'url': None
            },
            {
                'chunk_id': 'chunk_137a38aed9745593b694a86d0f21149f',
                'source_id': 'wilke_fundamentals',
                'source_name': 'wilke fundamentals',
                'title': 'redundant coding',
                'score': 0.714485,
                'source_path': 'wilke_fundamentals/dataviz_redundant-coding.txt',
                'url': None
            },
            {
                'chunk_id': 'chunk_1f06005e36db57a29d50d37b57f7ca5d',
                'source_id': 'from_data_to_viz',
                'source_name': 'from data to viz',
                'title': 'graph parallel',
                'score': 0.714219,
                'source_path': 'from_data_to_viz/graph_parallel.txt',
                'url': None
            },
            {
                'chunk_id': 'chunk_28e84461cb9a5e2f9590cec7c54ae4cc',
                'source_id': 'wi